In [1]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import seaborn as sns
import numpy as np
from scipy.stats import gaussian_kde

In [2]:
def load_timeline_items(json_path: str) -> list:
  """Loads a Google Timeline JSON export and returns its list of items.

  Handles both a direct list structure and one nested under a
  'timelineObjects' key.
  """
  with open(json_path, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

  return (
      raw_data
      if isinstance(raw_data, list)
      else raw_data.get("timelineObjects", raw_data)
  )


def extract_coordinates_from_semantic_json(json_path: str) -> pd.DataFrame:
  """Extracts latitude/longitude coordinates from a Google Timeline export.

  Reads 'visit -> topCandidate -> placeLocation' from each timeline item.

  Parameters
  ----------
  json_path : str
      Path to the Google Timeline JSON file.

  Returns
  -------
  pd.DataFrame
      DataFrame with 'latitude' and 'longitude' columns.
  """
  print(f"Loading Timeline dataset from: {json_path}...")
  timeline_items = load_timeline_items(json_path)

  extracted_points = []
  for item in timeline_items:
    visit_data = item.get("visit", {})
    top_candidate = visit_data.get("topCandidate", {})
    place_location = top_candidate.get("placeLocation", "")

    # Parse coordinates formatted as "geo:latitude,longitude"
    if place_location.startswith("geo:"):
      try:
        latitude_str, longitude_str = place_location.replace(
            "geo:", ""
        ).split(",")
        extracted_points.append((float(latitude_str), float(longitude_str)))
      except (ValueError, AttributeError):
        continue

  locations_df = pd.DataFrame(
      extracted_points, columns=["latitude", "longitude"]
  )

  # Guard against malformed coordinates outside valid geographic bounds
  valid_locations_df = locations_df[
      (locations_df["latitude"].between(-90, 90))
      & (locations_df["longitude"].between(-180, 180))
  ].copy()

  print(
      f"Successfully extracted {len(valid_locations_df)} valid location"
      " points."
  )
  return valid_locations_df

In [3]:
# Set your file path here
TIMELINE_JSON_PATH = "location-history.json"

# Extract location data
location_data_df = extract_coordinates_from_semantic_json(TIMELINE_JSON_PATH)

# Display the first few rows to verify extraction
location_data_df.head()

Loading Timeline dataset from: location-history.json...
Successfully extracted 16241 valid location points.


,latitude,longitude
0,4.659418,-74.108378
1,4.659418,-74.108378
2,4.654980,-74.055220
3,4.654391,-74.055318
4,4.725998,-74.057093


In [ ]:
# Output file configuration
OUTPUT_IMAGE_PATH = "heatmap_heightmap_americas.png"
FIGURE_DPI = 300
GRID_RESOLUTION = 1200  # High-definition resolution for focused map
DENSITY_GAMMA = 0.4  # < 1.0 compresses the dynamic range, lifting smaller
# cities/visits that would otherwise be flattened by the home peak
SECONDARY_GAMMA = 0.2  # Extra compression on top of DENSITY_GAMMA, pulling
# visited areas closer to the top of the range. Still a power law (not a
# linear floor), so true background (no visits) stays at exactly 0.0
# instead of being lifted off the ground.

# Define geographic boundaries specifically for the Americas
MIN_LON, MAX_LON = -170.0, -30.0  # Longitude bounds (West to East)
MIN_LAT, MAX_LAT = -60.0, 75.0  # Latitude bounds (South to North)

print("Filtering coordinates within the Americas bounding box...")

# Filter DataFrame points strictly within the Americas bounding box
americas_mask = (
    location_data_df["longitude"].between(MIN_LON, MAX_LON)
) & (location_data_df["latitude"].between(MIN_LAT, MAX_LAT))
filtered_df = location_data_df[americas_mask].copy()

lon_data = filtered_df["longitude"].values
lat_data = filtered_df["latitude"].values

print("Calculating 2D Kernel Density Estimation (KDE)... Please wait.")

# Create grid specifically covering the Americas region
lon_grid = np.linspace(MIN_LON, MAX_LON, GRID_RESOLUTION)
# Maintain aspect ratio based on coordinate span
lat_span_ratio = (MAX_LAT - MIN_LAT) / (MAX_LON - MIN_LON)
lat_grid = np.linspace(
    MIN_LAT, MAX_LAT, int(GRID_RESOLUTION * lat_span_ratio)
)
mesh_lon, mesh_lat = np.meshgrid(lon_grid, lat_grid)

# Evaluate Gaussian KDE on filtered points
data_positions = np.vstack([lon_data, lat_data])
grid_positions = np.vstack([mesh_lon.ravel(), mesh_lat.ravel()])

kde = gaussian_kde(data_positions)
# Adjust peak sharpness (0.35 - 0.4 works great for regional focus)
kde.set_bandwidth(bw_method=kde.factor * 0.35)

density_z = kde(grid_positions).reshape(mesh_lon.shape)

# Apply a power-law (gamma) transform before normalizing. Visit density is
# heavily skewed toward home/frequent locations, so a linear normalization
# leaves most other visited cities nearly invisible (<1% of peak height).
# Raising to a power < 1.0 compresses that dynamic range, boosting smaller
# peaks while keeping the tallest one on top.
density_compressed = np.power(density_z, DENSITY_GAMMA)

# Normalize strictly between 0.0 and 1.0
density_unit = (density_compressed - density_compressed.min()) / (
    density_compressed.max() - density_compressed.min()
)

# Apply a second compression pass for extra softening. Power laws always
# map 0 -> 0 and 1 -> 1, so this pulls visited areas further up toward the
# top of the range without disturbing true background (still exactly
# 0.0) or creating a hard edge at the boundary of each visited area.
density_z_normalized = np.power(density_unit, SECONDARY_GAMMA)

print("Rendering Heightmap texture for the Americas...")

# Compute figure aspect ratio dynamically
fig_width = 16.0
fig_height = fig_width * lat_span_ratio

fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=FIGURE_DPI)

fig.patch.set_facecolor("black")
ax.set_facecolor("black")

# Set exact axes limits
ax.set_xlim(MIN_LON, MAX_LON)
ax.set_ylim(MIN_LAT, MAX_LAT)

# Display density image mapped to Americas bounds
ax.imshow(
    density_z_normalized,
    cmap="gray",
    extent=[MIN_LON, MAX_LON, MIN_LAT, MAX_LAT],
    origin="lower",
    aspect="equal",
    vmin=0.0,
    vmax=1.0,
)

plt.axis("off")
plt.subplots_adjust(top=1, bottom=0, right=1, left=0, hspace=0, wspace=0)

# Save focused Heightmap texture
plt.savefig(
    OUTPUT_IMAGE_PATH,
    bbox_inches="tight",
    pad_inches=0,
    facecolor="black",
    dpi=FIGURE_DPI,
)
plt.show()

print(
    f"Americas Heightmap texture successfully saved at: {OUTPUT_IMAGE_PATH}"
)

In [ ]:
REFERENCE_IMAGE_PATH = "heatmap_americas_reference.png"
FIGURE_DPI = 300

print("Loading world boundaries for Americas validation...")
WORLD_URL = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(WORLD_URL)

# Compute canvas aspect ratio dynamically
lat_span_ratio = (MAX_LAT - MIN_LAT) / (MAX_LON - MIN_LON)
fig_width = 16.0
fig_height = fig_width * lat_span_ratio

fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=FIGURE_DPI)

fig.patch.set_facecolor("#080c14")
ax.set_facecolor("#080c14")

# Set boundaries restricted to the Americas
ax.set_xlim(MIN_LON, MAX_LON)
ax.set_ylim(MIN_LAT, MAX_LAT)

# 1. Plot Country Borders
world.boundary.plot(ax=ax, color="#2A364F", linewidth=0.7, zorder=1)
world.plot(ax=ax, color="#101826", zorder=0)

# 2. Overlay Heatmap
masked_density = np.ma.masked_where(
    density_z_normalized < 0.01, density_z_normalized
)

ax.imshow(
    masked_density,
    cmap="magma",
    extent=[MIN_LON, MAX_LON, MIN_LAT, MAX_LAT],
    origin="lower",
    aspect="equal",
    alpha=0.85,
    zorder=2,
)

# 3. Add Raw Data Points
ax.scatter(
    filtered_df["longitude"],
    filtered_df["latitude"],
    color="#00F0FF",
    s=1.0,
    alpha=0.5,
    zorder=3,
)

plt.title(
    "Google Timeline Heatmap - Americas Regional Verification",
    color="white",
    fontsize=14,
    pad=15,
)
plt.axis("off")
plt.subplots_adjust(top=0.95, bottom=0, right=1, left=0, hspace=0, wspace=0)

plt.savefig(
    REFERENCE_IMAGE_PATH,
    bbox_inches="tight",
    pad_inches=0.1,
    facecolor="#080c14",
    dpi=FIGURE_DPI,
)
plt.show()

print(
    f"Americas validation overlay saved successfully at:"
    f" {REFERENCE_IMAGE_PATH}"
)

In [6]:
# Identify the principal cities visited within the Americas bbox. Uses
# offline reverse geocoding (most Timeline entries have no usable place
# name), clusters nearby visits into metro areas, and labels each cluster
# with the largest known city nearby rather than the closest small
# locality. Keeps only the most recent visit per city.
import reverse_geocoder as rg
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

METRO_RADIUS_KM = 50.0  # Visits within this range are treated as one place
MAJOR_CITY_POPULATION_THRESHOLD = 100_000  # Min. population to count as a "principal" city
EARTH_RADIUS_KM = 6371.0
GEONAMES_MAJOR_CITIES_URL = "https://download.geonames.org/export/dump/cities15000.zip"

timeline_items = load_timeline_items(TIMELINE_JSON_PATH)

# Collect every visit's coordinates + timestamp within the Americas bbox
visit_records = []
for item in timeline_items:
  visit_data = item.get("visit", {})
  top_candidate = visit_data.get("topCandidate", {})
  place_location = top_candidate.get("placeLocation", "")
  start_time_str = item.get("startTime", "")

  if not place_location.startswith("geo:"):
    continue

  try:
    latitude_val, longitude_val = map(
        float, place_location.replace("geo:", "").split(",")
    )
  except (ValueError, AttributeError):
    continue

  if not (
      MIN_LON <= longitude_val <= MAX_LON and MIN_LAT <= latitude_val <= MAX_LAT
  ):
    continue

  visit_records.append({
      "latitude": latitude_val,
      "longitude": longitude_val,
      "visit_time": pd.to_datetime(start_time_str, errors="coerce", utc=True),
  })

visits_df = pd.DataFrame(visit_records).dropna(subset=["visit_time"])
print(f"Collected {len(visits_df)} visit points within the Americas bbox.")


def to_unit_sphere_xyz(lat_deg: np.ndarray, lon_deg: np.ndarray) -> np.ndarray:
  """Projects lat/lon degrees onto a unit sphere for Euclidean distance
  queries (cKDTree doesn't support great-circle distance directly)."""
  lat_rad, lon_rad = np.radians(lat_deg), np.radians(lon_deg)
  return np.column_stack([
      np.cos(lat_rad) * np.cos(lon_rad),
      np.cos(lat_rad) * np.sin(lon_rad),
      np.sin(lat_rad),
  ])


def km_to_chord_distance(distance_km: float) -> float:
  """Converts a great-circle distance to the equivalent straight-line
  (chord) distance between two points on the unit sphere."""
  return 2 * np.sin((distance_km / EARTH_RADIUS_KM) / 2)


print("Reverse geocoding visit points to their nearest locality (offline lookup)...")
coordinates = list(zip(visits_df["latitude"], visits_df["longitude"]))
# mode=1 forces single-threaded lookup, avoiding multiprocessing issues
# inside Jupyter kernels
geocode_results = rg.search(coordinates, mode=1)

visits_df["city"] = [result["name"] for result in geocode_results]
visits_df["state"] = [result["admin1"] for result in geocode_results]
visits_df["country"] = [result["cc"] for result in geocode_results]

# --- Cluster nearby visits into metro areas ---------------------------
# Group visit points within METRO_RADIUS_KM of each other so a city and
# its surrounding suburbs are treated as a single visited place.
visit_xyz = to_unit_sphere_xyz(
    visits_df["latitude"].to_numpy(), visits_df["longitude"].to_numpy()
)
metro_eps = km_to_chord_distance(METRO_RADIUS_KM)

visit_tree = cKDTree(visit_xyz)
close_pairs = visit_tree.query_pairs(r=metro_eps, output_type="ndarray")

num_points = len(visits_df)
if len(close_pairs) > 0:
  row_idx = np.concatenate([close_pairs[:, 0], close_pairs[:, 1]])
  col_idx = np.concatenate([close_pairs[:, 1], close_pairs[:, 0]])
  adjacency = coo_matrix(
      (np.ones(len(row_idx)), (row_idx, col_idx)),
      shape=(num_points, num_points),
  )
  _, cluster_labels = connected_components(adjacency, directed=False)
else:
  cluster_labels = np.arange(num_points)

visits_df["metro_cluster"] = cluster_labels

# --- Load a "principal cities" reference so clusters are labeled with the
# largest nearby city instead of the closest small locality -------------
print(
    "Downloading GeoNames major cities database (population >="
    f" {MAJOR_CITY_POPULATION_THRESHOLD:,})..."
)
geonames_columns = [
    "geonameid", "name", "asciiname", "alternatenames", "latitude",
    "longitude", "feature_class", "feature_code", "country_code", "cc2",
    "admin1_code", "admin2_code", "admin3_code", "admin4_code",
    "population", "elevation", "dem", "timezone", "modification_date",
]
major_cities_df = pd.read_csv(
    GEONAMES_MAJOR_CITIES_URL,
    sep="\t",
    header=None,
    names=geonames_columns,
    usecols=["name", "latitude", "longitude", "country_code", "population"],
    compression="zip",
)
major_cities_df = major_cities_df[
    (major_cities_df["population"] >= MAJOR_CITY_POPULATION_THRESHOLD)
    & (major_cities_df["longitude"].between(MIN_LON, MAX_LON))
    & (major_cities_df["latitude"].between(MIN_LAT, MAX_LAT))
].reset_index(drop=True)

# Reverse geocode the major cities themselves so their state/province name
# uses the same readable format as the visit lookups above
major_geocode_results = rg.search(
    list(zip(major_cities_df["latitude"], major_cities_df["longitude"])),
    mode=1,
)
major_cities_df["state"] = [r["admin1"] for r in major_geocode_results]

major_xyz = to_unit_sphere_xyz(
    major_cities_df["latitude"].to_numpy(),
    major_cities_df["longitude"].to_numpy(),
)
major_tree = cKDTree(major_xyz)
print(f"{len(major_cities_df)} major cities loaded for the Americas region.")


def summarize_metro_cluster(group: pd.DataFrame) -> pd.Series:
  """Collapses a cluster of nearby visits into one representative city row.

  Labels the cluster with the largest known city within METRO_RADIUS_KM of
  its centroid; falls back to the most-visited local place name if no
  major city is nearby.
  """
  cluster_centroid_xyz = visit_xyz[group.index.to_numpy()].mean(axis=0)
  cluster_centroid_xyz /= np.linalg.norm(cluster_centroid_xyz)

  nearby_major_idx = major_tree.query_ball_point(
      cluster_centroid_xyz, r=metro_eps
  )
  most_recent_visit = group.sort_values("visit_time", ascending=False).iloc[0]

  if nearby_major_idx:
    nearby_majors = major_cities_df.iloc[nearby_major_idx]
    principal_city = nearby_majors.loc[nearby_majors["population"].idxmax()]
    city_name = principal_city["name"]
    state_name = principal_city["state"]
    country_code = principal_city["country_code"]
  else:
    most_visited_city = group["city"].mode().iloc[0]
    representative = group[group["city"] == most_visited_city].iloc[0]
    city_name = most_visited_city
    state_name = representative["state"]
    country_code = representative["country"]

  return pd.Series({
      "city": city_name,
      "state": state_name,
      "country": country_code,
      "latitude": most_recent_visit["latitude"],
      "longitude": most_recent_visit["longitude"],
      "last_visit": most_recent_visit["visit_time"],
      "visit_count": len(group),
  })


visited_cities_df = (
    visits_df.groupby("metro_cluster")
    .apply(summarize_metro_cluster, include_groups=False)
    .reset_index(drop=True)
    .sort_values("last_visit", ascending=False)
    .reset_index(drop=True)
)

# Export to CSV for downstream use in Blender (see README Step 4)
OUTPUT_CITIES_CSV_PATH = "visited_cities.csv"
visited_cities_df.to_csv(OUTPUT_CITIES_CSV_PATH, index=False)

print(
    f"✅ Identified {len(visited_cities_df)} visited cities/metro areas."
    f" Saved to {OUTPUT_CITIES_CSV_PATH}"
)
visited_cities_df.head(20)

Collected 16241 visit points within the Americas bbox.
Reverse geocoding visit points to their nearest locality (offline lookup)...
Loading formatted geocoded file...
1366 major cities loaded for the Americas region.
✅ Identified 54 visited cities/metro areas. Saved to visited_cities.csv


,city,state,country,latitude,longitude,last_visit,visit_count
0,Bogotá,Bogota D.C.,CO,4.742654,-74.056415,2026-08-12 17:29:48.267000+00:00,13694
1,Tampa,Florida,US,27.980365,-82.535403,2026-08-07 17:21:58.558000+00:00,114
2,Orlando,Florida,US,28.370568,-81.519359,2026-08-05 16:42:18+00:00,81
3,Chicago,Illinois,US,41.980259,-87.908986,2026-06-08 00:36:41.928000+00:00,63
4,Panama City,Panama,PA,9.067390,-79.386346,2025-12-21 17:14:13.019000+00:00,1530
5,Mobile,Alabama,US,30.634232,-87.676635,2024-06-26 15:32:37.012000+00:00,3
6,Baton Rouge,Louisiana,US,30.451890,-90.957245,2024-06-26 00:52:25.139000+00:00,3
7,Houston,Texas,US,29.732680,-94.957536,2024-06-25 19:57:46.291000+00:00,17
8,Beaumont,Texas,US,30.130442,-93.882513,2024-06-23 16:23:38+00:00,1
9,New Orleans,Louisiana,US,29.994400,-90.154395,2024-06-23 05:32:08.220000+00:00,18
